## ESPnet TTS Demonstracija

Šie skriptai demonstruoja kaip galima sintezuoti lietuvišką garsą, kai lokaliai turime:
1. Akustinio modelio archyvą
2. Vokoderio archyvą
   
### 0. Inicializuokite pagalbines funkcijas

In [ ]:
# Inicializuokite pagalbines funkcijas

from espnet2.bin.tts_inference import Text2Speech
import soundfile as sf
from IPython.display import Audio, HTML
from espnet2.main_funcs.pack_funcs import unpack
import tarfile
from pathlib import Path

cache_dir = "./.cache/"
device = "cpu"

def _safe_extract_tar(tar: tarfile.TarFile, path: Path):
    base = path.resolve()

    for member in tar.getmembers():
        member_path = (path / member.name).resolve()
        if not str(member_path).startswith(str(base)):
            raise Exception("Unsafe tar archive")

    tar.extractall(path)

def find_vocoder(search_dir: Path) -> str:
    for file in search_dir.rglob("*.pkl"):
        # print(f"found {str(file)}")
        return str(file)

    raise FileNotFoundError(f"No *.pkl file found in the {search_dir}")    


def extract_archive_cached(archive_path: str, cache_dir: str) -> Path:
    archive_path = Path(archive_path)
    cache_dir = Path(cache_dir)

    stat = archive_path.stat()

    # fast cache key (no hashing)
    key = f"{archive_path.name}_{stat.st_size}_{int(stat.st_mtime)}"

    extract_path = cache_dir / key
    marker = extract_path / ".done"

    if marker.exists():
        return extract_path

    extract_path.mkdir(parents=True, exist_ok=True)

    if archive_path.name.endswith(".tar.gz"):
        with tarfile.open(archive_path) as tar:
            _safe_extract_tar(tar, extract_path)
    else:
        raise ValueError(f"Unsupported archive format: {archive_path}")

    marker.touch()
    return extract_path
    

def init_tts(am_file=None, vocoder_file=None):
    if not vocoder_file:
        vocoder_file = None # make sure it's None, not empty string
    else:
        vocoder_dir = extract_archive_cached(archive_path=vocoder_file, cache_dir=cache_dir)
        # print(f"vocoder_dir: {vocoder_dir}")
        vocoder_file = find_vocoder(vocoder_dir)

    print(f"\n===================================\nAM      = {am_file}")
    print(f"Vocoder = {vocoder_file if vocoder_file else 'Griffin-Lim'}")
    print(f"===================================")

    kwargs = unpack(input_archive = am_file, outpath = cache_dir)
    # print (kwargs)
    
    tts = Text2Speech.from_pretrained(
        vocoder_file=vocoder_file, 
        device=device,
        **kwargs
    )
    return tts

print ("ESPnet inicializuotas")    

### 1. Modelis

Nurodykite kelią iki Akustinio modelio ir Vokoderio archyvo

In [ ]:
# Nurodykite kelią iki akustinio modelio zip failo
am_zip_file="agn.tts_train_fastspeech2_raw_phn_espeak_ng_lt_train.loss.ave.zip"

# Nurodykite kelią iki vokoderio tar gz failo
# arba palikite tuščią, jei norite naudoti Griffin-Lim algoritmą
vocoder_tar_gz_file="agn.style.v01-600000.tar.gz"
# vocoder_tar_gz_file=None

# bus išskleisti į './.cache/' katalogą"
print (f"AM zip failas  : {am_zip_file}")
print (f"Vocoder failas : {vocoder_tar_gz_file}")

# Inicializuokite TTS modelį
tts = init_tts(am_file=am_zip_file, vocoder_file=vocoder_tar_gz_file)
print (f"\n\nREADY: modelis įkeltas\n") 

### 2. Sintezavimas

Pakoreguokite/įveskite sakinius, kuriuos norite perskaityti

In [ ]:
### Kai kurie tekstai paimti iš lrt.lt
texts = ["Sveiki, aš naujas lietuviškas balsas.", 
         "Aš esu labai gerai įrašytas.", 
         "Kaip Jums patinku?", 
         "Pernai maitinimo sektorių sukrėtė dešimtmečius veikusių restoranų ir kavinių bankrotai.", 
         "Šiomis dienomis orai Lietuvoje ims šilti, kris šlapdriba, sniegas, reikės pasisaugoti lijundros.", 
         "Tai savo „Facebook“ paskyroje sekmadienį vakare pranešė Ukrainos pirmasis vicepremjeras ir energetikos ministras Denysas Šmyhalis, skelbia „Ukrinform“."
        ]
for i, text in enumerate(texts):
    print(f"\n==========================================================================\nText = {text}")
    wav = tts(text)["wav"]
    filename = f"output_{i}.wav"
    sf.write(filename, wav.numpy(), tts.fs)
    display(Audio(filename))
